# Single MEI IIIF + measure-zone integration (Workflow 1)

Use this Workflow 1 maintainer notebook when a single retyped MEI file does not already follow the BSB/IIIF filename convention. The reusable pipeline functions come from the installed `camat` package; corpus inputs and generated files stay in the external edition repository. The notebook stages the file under the expected stem, downloads the matching BSB IIIF page image, sends that image to the measure detector, integrates measure zones, and writes the final `*_facs_zones.mei` output. Network and write steps are opt-in.

Example target: `Demo/70_musicxml-zip_verovio.mei` comes from the archive page <https://digitale-sammlungen.de/en/view/bsb00023199?page=70>, so the staged source should become `bsb00023199_00070.mei` and the final output should become `bsb00023199_00070_facs_zones.mei`.

In [ ]:
from __future__ import annotations

from pathlib import Path
from urllib.parse import parse_qs, urlparse
import re
import shutil

from camat import (
    DETECTOR_URL,
    IIIF_IMAGE_URL_TEMPLATE,
    detect_and_integrate_mei as process_mei_file,
    download_facsimile_images as download_images,
    find_image_for_stem,
    get_body_measures,
    parse_graphic_from_output_mei,
    parse_mei,
    read_image_size,
    sha256_bytes,
    sha256_file,
)
import requests

REPO_ROOT = Path.cwd().resolve()

## 1. Configure the single-file job

Set `SOURCE_MEI` to the retyped file and `ARCHIVE_URL` to the BSB page. `SOURCE_MEI` can be a normal repo-relative string such as `"Demo/26_musescore_verovio.mei"`, an absolute path string, or a `Path`. The archive URL is used to derive the target IIIF stem automatically.

In [ ]:
SOURCE_MEI = "Demo/Buxtehude-Anhang-S._185_musicxml_verovio.mei"
ARCHIVE_URL = "https://digitale-sammlungen.de/en/view/bsb00023199?page=185"

# Usually keep the staged source and final output beside the retyped source.
# Set this to a path string if you want outputs somewhere else.
TARGET_DIR = None

# The existing corpus pipeline defaults.
TARGET_DPI = 500
PAGE_WIDTH_MM = 210.0
MINIMUM_MEASURES = 5
MAX_MEASURE_MISMATCH = 1
TIMEOUT = 180

# Network calls and writes are deliberately disabled until this is set to True.
RUN_IIIF_INTEGRATION = False

# Set these explicitly when repeating a step.
OVERWRITE_STAGED_MEI = False
OVERWRITE_OUTPUT = False
REUSE_ANNOTATIONS = False

In [ ]:
def derive_bsb_stem(archive_url: str) -> tuple[str, int, str]:
    parsed = urlparse(archive_url)
    match = re.search(r"/view/(bsb\d+)", parsed.path)
    if not match:
        raise ValueError(f"Could not find a BSB id in {archive_url!r}")

    query = parse_qs(parsed.query)
    if "page" not in query or not query["page"]:
        raise ValueError(f"Could not find a page= value in {archive_url!r}")

    bsb_id = match.group(1)
    page = int(query["page"][0])
    stem = f"{bsb_id}_{page:05d}"
    return bsb_id, page, stem


def resolve_repo_path(path: str | Path | None) -> Path | None:
    if path is None:
        return None
    resolved = Path(path).expanduser()
    if not resolved.is_absolute():
        resolved = REPO_ROOT / resolved
    return resolved.resolve()


def display_path(path: Path) -> str:
    try:
        return str(path.relative_to(REPO_ROOT))
    except ValueError:
        return str(path)


def stage_single_mei(source_mei: Path, target_dir: Path, target_stem: str, *, overwrite: bool) -> Path:
    source_mei = source_mei.resolve()
    target_dir.mkdir(parents=True, exist_ok=True)
    staged_mei = target_dir / f"{target_stem}.mei"

    if not source_mei.is_file():
        raise FileNotFoundError(source_mei)
    if staged_mei.exists() and staged_mei.resolve() != source_mei and not overwrite:
        raise FileExistsError(
            f"Staged MEI already exists: {staged_mei}. Set OVERWRITE_STAGED_MEI = True to replace it."
        )
    if staged_mei.resolve() != source_mei:
        shutil.copy2(source_mei, staged_mei)
    return staged_mei


def count_measures(mei_path: Path) -> int:
    return len(get_body_measures(parse_mei(str(mei_path)).getroot()))


SOURCE_MEI = resolve_repo_path(SOURCE_MEI)
TARGET_DIR = resolve_repo_path(TARGET_DIR) or SOURCE_MEI.parent

bsb_id, page_number, target_stem = derive_bsb_stem(ARCHIVE_URL)
staged_mei = TARGET_DIR / f"{target_stem}.mei"
image_dir = TARGET_DIR / "img"
annotation_path = TARGET_DIR / f"{target_stem}_measure_annotations.xml"
final_mei = TARGET_DIR / f"{target_stem}_facs_zones.mei"

print(f"BSB id:       {bsb_id}")
print(f"Page number:  {page_number}")
print(f"IIIF stem:    {target_stem}")
print(f"Source MEI:   {display_path(SOURCE_MEI)}")
print(f"Staged MEI:   {display_path(staged_mei)}")
print(f"Annotation:   {display_path(annotation_path)}")
print(f"Final output: {display_path(final_mei)}")

## 2. Stage the MEI under the IIIF stem

The folder scripts infer IIIF URLs from the MEI filename stem. This cell copies the retyped file to the expected name and leaves the original untouched.

In [ ]:
if not RUN_IIIF_INTEGRATION:
    raise RuntimeError("Review the paths and settings, then set RUN_IIIF_INTEGRATION = True")

staged_mei = stage_single_mei(
    SOURCE_MEI,
    TARGET_DIR,
    target_stem,
    overwrite=OVERWRITE_STAGED_MEI,
)

measure_count = count_measures(staged_mei)
print(f"Staged {display_path(SOURCE_MEI)} -> {display_path(staged_mei)}")
print(f"Source measure count: {measure_count}")

## 3. Download the matching IIIF facsimile image

This uses the same BSB IIIF image template as the folder pipeline. The local image is saved as `TARGET_DIR/img/<stem>.jpg` so the detector integration can find it.

In [ ]:
downloaded, skipped, failed = download_images(
    mei_files=[staged_mei],
    output_dir=image_dir,
    width=None,
    quality_rank=None,
    target_dpi=TARGET_DPI,
    page_width_mm=PAGE_WIDTH_MM,
    timeout=TIMEOUT,
    delay=0,
    overwrite=False,
)
if failed:
    raise RuntimeError("Facsimile download failed")

image_path = find_image_for_stem(image_dir, target_stem)
width, height = read_image_size(image_path)
expected_iiif_url = IIIF_IMAGE_URL_TEMPLATE.format(stem=target_stem, width=width)

print(f"Downloaded={downloaded}, skipped={skipped}")
print(f"Image: {display_path(image_path)} ({width} x {height})")
print(expected_iiif_url)

## 4. Upload/detect and integrate local facsimile zones

This writes/reuses `<stem>_measure_annotations.xml` and creates `<stem>_facs_zones.mei` with a local `img/...` graphic target. If the detector already produced an annotation file, keep `REUSE_ANNOTATIONS = True` to avoid another upload.

In [ ]:
annotation_path, local_output = process_mei_file(
    staged_mei,
    image_dir=image_dir,
    detector_url=DETECTOR_URL,
    timeout=TIMEOUT,
    retries=2,
    retry_delay=2.0,
    minimum_measures=MINIMUM_MEASURES,
    max_measure_mismatch=MAX_MEASURE_MISMATCH,
    annotation_suffix="_measure_annotations.xml",
    output_suffix="_facs_zones",
    reuse_annotations=REUSE_ANNOTATIONS,
    overwrite=OVERWRITE_OUTPUT,
    graphic_target_mode="local",
    iiif_url_template=IIIF_IMAGE_URL_TEMPLATE,
)

print(f"Annotation: {display_path(annotation_path)}")
print(f"Local output: {display_path(local_output)}")

## 5. Validate local image against the IIIF URL

This checks that the local image bytes match the BSB IIIF URL implied by the staged filename and downloaded width.

In [ ]:
local_hash = sha256_file(image_path)
response = requests.get(expected_iiif_url, timeout=TIMEOUT)
response.raise_for_status()
remote_hash = sha256_bytes(response.content)

if local_hash != remote_hash:
    raise RuntimeError(f"IIIF mismatch: local={local_hash}, remote={remote_hash}")

print("Local facsimile matches IIIF source")
print(local_hash)

## 6. Re-integrate with the final IIIF graphic target

The publication-ready output keeps the same `_facs_zones` suffix but replaces the local `img/...` target with the BSB IIIF URL.

In [ ]:
annotation_path, final_mei = process_mei_file(
    staged_mei,
    image_dir=image_dir,
    detector_url=DETECTOR_URL,
    timeout=TIMEOUT,
    retries=2,
    retry_delay=2.0,
    minimum_measures=MINIMUM_MEASURES,
    max_measure_mismatch=MAX_MEASURE_MISMATCH,
    annotation_suffix="_measure_annotations.xml",
    output_suffix="_facs_zones",
    reuse_annotations=True,
    overwrite=True,
    graphic_target_mode="iiif",
    iiif_url_template=IIIF_IMAGE_URL_TEMPLATE,
)

print(f"Final output: {display_path(final_mei)}")

## 7. Final checks

Confirm that the final file has the `_facs_zones` suffix, uses the expected IIIF URL, and contains matching graphic dimensions.

In [ ]:
target, mei_width, mei_height = parse_graphic_from_output_mei(final_mei)

assert final_mei.name == f"{target_stem}_facs_zones.mei"
assert target == expected_iiif_url, (target, expected_iiif_url)
assert (mei_width, mei_height) == (width, height), ((mei_width, mei_height), (width, height))

print("Final MEI is ready")
print(f"MEI:    {display_path(final_mei)}")
print(f"Target: {target}")
print(f"Size:   {mei_width} x {mei_height}")

## Notes for the next single file

1. Change `SOURCE_MEI` and `ARCHIVE_URL`. `SOURCE_MEI = "Demo/26_musescore_verovio.mei"` is enough for a file inside the repo.
2. Review every path and overwrite flag, then set `RUN_IIIF_INTEGRATION = True`.
3. Run cells top to bottom.
4. If the staged `bsb..._00000.mei` already exists and should be replaced, set `OVERWRITE_STAGED_MEI = True`.
5. If measure detection needs to be rerun, delete the old `<stem>_measure_annotations.xml` or set `REUSE_ANNOTATIONS = False` and `OVERWRITE_OUTPUT = True`.